In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Completely uninstall and then reinstall a specific version of meteostat
# Also, print the version to confirm the installation.
!pip uninstall meteostat -y
!pip install meteostat==1.7.6
import meteostat
print(f"Meteostat version installed: {meteostat.__version__}")

from pathlib import Path

import numpy as np
import pandas as pd

from datetime import datetime

from meteostat import Point, Hourly

Meteostat version installed: 1.7.6


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import drive
import os
import shutil
import time # Added for small delays

mountpoint = '/content/drive'

# --- Step 1: Attempt to unmount if already mounted ---
if os.path.ismount(mountpoint):
    print(f"Detected {mountpoint} is already mounted. Attempting to unmount...")
    try:
        !fusermount -uz {mountpoint}
        print(f"Successfully unmounted {mountpoint} (if it was a fusermount). This clears any temporary files on the Colab VM's local filesystem at this location, not your actual Google Drive files.")
    except Exception as e:
        print(f"Error during fusermount unmount attempt: {e}. Proceeding with directory cleanup.")
else:
    print(f"{mountpoint} is not currently an active mount point.")

# Add a small delay to ensure unmount operations fully propagate
time.sleep(1)

# --- Step 2: Forcibly remove and recreate the mountpoint directory to ensure it's empty ---
print(f"Ensuring {mountpoint} is a fresh, empty directory before mounting...")
if os.path.exists(mountpoint):
    try:
        shutil.rmtree(mountpoint)
        print(f"Successfully removed existing {mountpoint} directory and its contents from the Colab VM's local filesystem.")
        time.sleep(0.5) # Give a moment for filesystem to register removal
    except OSError as e:
        print(f"Error removing {mountpoint}: {e}. This indicates a persistent lock or permission issue.")
        print("Attempting to create directory, but mount might still fail if directory is locked.")
else:
    print(f"Directory {mountpoint} does not exist on the Colab VM, creating it.")

# Always create the mountpoint directory to ensure it's an empty, clean target
# This ensures it's empty right before drive.mount is called
os.makedirs(mountpoint, exist_ok=True)
print(f"Created/ensured empty mountpoint directory: {mountpoint} on the Colab VM.")
time.sleep(0.5) # Another small delay

# --- Step 3: Attempt to mount Google Drive ---
print("Attempting to mount Google Drive...")
try:
    drive.mount(mountpoint, force_remount=True)
    print("Google Drive mount attempt complete.")
    if os.path.ismount(mountpoint):
        print("Google Drive successfully mounted! Your files should now be accessible via this path.")
    else:
        print("Google Drive did not mount successfully, despite no immediate exception.")
except Exception as e:
    print(f"An error occurred during drive.mount: {e}")
    print("Please check your permissions, restart the runtime, or ensure no other process is using the mountpoint.")

from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras import layers, models

/content/drive is not currently an active mount point.
Ensuring /content/drive is a fresh, empty directory before mounting...
Directory /content/drive does not exist on the Colab VM, creating it.
Created/ensured empty mountpoint directory: /content/drive on the Colab VM.
Attempting to mount Google Drive...
Mounted at /content/drive
Google Drive mount attempt complete.
Google Drive successfully mounted! Your files should now be accessible via this path.


In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/RideHailingPhD")

DATASET_DIR = PROJECT_ROOT / "datasets"

WEATHER_DIR = DATASET_DIR / "weather"

RAW_DIR = WEATHER_DIR / "raw"

PROCESSED_DIR = WEATHER_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
YEAR = 2026

MONTH = 1

CITY_NAME = "New York City"

LATITUDE = 40.7128
LONGITUDE = -74.0060
ALTITUDE = 10

In [ ]:
location = Point(
    LATITUDE,
    LONGITUDE,
    ALTITUDE
)

start = datetime(YEAR, MONTH, 1)

end = datetime(YEAR, MONTH, 31, 23, 59)

weather = Hourly(
    location,
    start,
    end
).fetch()

In [ ]:
print(weather.shape)

display(weather.head())

print(weather.columns)

(744, 11)


,temp,dwpt,rhum,prcp,snow,wdir,wspd,wpgt,pres,tsun,coco
time,,,,,,,,,,,
2026-01-01 00:00:00,-0.0,-7.3,58.0,0.0,<NA>,230.0,17.0,19.0,1009.0,<NA>,1.0
2026-01-01 01:00:00,1.0,-6.3,58.0,0.0,<NA>,220.0,17.0,<NA>,1008.0,<NA>,3.0
2026-01-01 02:00:00,1.0,-6.3,58.0,0.0,<NA>,220.0,17.0,<NA>,1007.0,<NA>,3.0
2026-01-01 03:00:00,1.0,-5.7,61.0,0.0,<NA>,220.0,15.0,18.0,1006.0,<NA>,3.0
2026-01-01 04:00:00,1.0,-5.7,61.0,0.0,<NA>,200.0,19.0,25.0,1005.0,<NA>,3.0


Index(['temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres',
       'tsun', 'coco'],
      dtype='object')


In [ ]:
raw_file = (
    RAW_DIR /
    f"nyc_weather_{YEAR}_{MONTH:02d}.parquet"
)

weather.to_parquet(raw_file)

print(raw_file)

/content/drive/MyDrive/RideHailingPhD/datasets/weather/raw/nyc_weather_2026_01.parquet


In [ ]:
weather = weather.reset_index()

weather = weather[
    [
        "time",
        "temp",
        "wspd",
        "coco"
    ]
]

In [ ]:
weather = weather.rename(
    columns={
        "time": "Datetime",
        "temp": "Temperature",
        "wspd": "WindSpeed",
        "coco": "WeatherCode"
    }
)

In [ ]:
weather = weather.sort_values("Datetime")

weather = weather.ffill()

weather = weather.bfill()

In [ ]:
processed_file = (
    PROCESSED_DIR /
    f"nyc_weather_processed_{YEAR}_{MONTH:02d}.parquet"
)

weather.to_parquet(
    processed_file,
    index=False
)

print(processed_file)

display(weather.head())

/content/drive/MyDrive/RideHailingPhD/datasets/weather/processed/nyc_weather_processed_2026_01.parquet


,Datetime,Temperature,WindSpeed,WeatherCode
0,2026-01-01 00:00:00,-0.0,17.0,1.0
1,2026-01-01 01:00:00,1.0,17.0,3.0
2,2026-01-01 02:00:00,1.0,17.0,3.0
3,2026-01-01 03:00:00,1.0,15.0,3.0
4,2026-01-01 04:00:00,1.0,19.0,3.0
